In [2]:
from helper import titanic_conn, q
conn = titanic_conn()

In [3]:
sql = '''
SELECT Name, Sex, Age 
from titanic 
where age < 5 and Pclass=1
limit 5
'''
q(conn, sql)

,Name,Sex,Age
0,"Allison, Miss. Helen Loraine",female,2.00
1,"Allison, Master. Hudson Trevor",male,0.92
2,"Dodge, Master. Washington",male,4.00


In [4]:
sql = '''
SELECT Name, Fare, Sex, Age
from titanic 
order by Fare DESC limit 5
'''
# q(conn, sql)

In [5]:
sql = '''
SELECT DISTINCT Embarked FROM titanic
'''
# q(conn, sql)

In [6]:
sql = '''
SELECT count(*) from titanic
where Name LIKE '%Miss%'
'''
# q(conn, sql)

In [7]:
sql = '''
SELECT count(*) from titanic 
where Embarked in ('C', 'Q')
'''
# q(conn, sql)

In [8]:
sql = '''
SELECT count(*) from titanic 
where Age Between 20 AND 30
'''
q(conn, sql)

,count(*)
0,245


In [9]:
# 문제1. 가장 나이가 많은 승객의 이름과 나이
# 문제2. 2등 3등석 (Pclass) 의 여성 생존자 수


In [10]:
# 문제1. 가장 나이가 많은 승객의 이름과 나이
sql = '''
-- SELECT Name, Max(age) from titanic 

SELECT Name, Age from titanic order by age DESC limit 1
'''
q(conn, sql)

,Name,Age
0,"Barkworth, Mr. Algernon Henry Wilson",80.0


In [11]:
# 문제2. 2등 3등석 (Pclass) 의 여성 생존자 수
sql = '''
select count(*) 
from titanic
where Survived = 1 AND Sex = 'female' AND Pclass = 3
'''
q(conn, sql)


,count(*)
0,72


In [12]:
sql = '''
SELECT count(*) as n,
    avg(Age) as avg_age,
    Min(Fare) as min_fare,
    Max(Fare) as max_fare,
    Sum(Survived) as survived,
    Sum(survived) / cast(count(*) as double) as propotion
from titanic
group by Pclass
having count(*) >= 200
'''
q(conn, sql)

,n,avg_age,min_fare,max_fare,survived,propotion
0,216,38.233441,0.0,512.3292,136,0.629630
1,491,25.140620,0.0,69.5500,119,0.242363


# Join

In [13]:
from helper import chinook_conn, q, show_tables, show_columns
conn = chinook_conn()
print(show_tables(conn))

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [14]:
show_columns(conn, 'Album')

,name,type
0,AlbumId,INTEGER
1,Title,NVARCHAR(160)
2,ArtistId,INTEGER


In [15]:
sql = 'select * From Album'
sql = '''
select t.Name as track_name, al.Title as album_title, ar.Name as artist_title
From Track t
join Album al on t.AlbumId = al.AlbumId
join Artist ar on al.ArtistId = ar.ArtistId
-- join 뭔가 on al.X = ar.X
Limit 5
'''
q(conn, sql)

# "From Track" join Album join Artist 
# t.Name , al.Title,, ar.Name 

# 문제. 장르별 곡 수 있는데, 가장 곡수가 많은 장르 5개. 
sql = '''
select g.Name as genre, count(*)  as n_tracks
from Track t
JOIN Genre g on t.GenreId = g.GenreId
Group by g.Name
Order by n_tracks DESC
limit 5
'''
q(conn, sql)

,genre,n_tracks
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130


In [16]:
sql = '''
SELECT Avg(Total) as avg_total from Invoice
'''
q(conn, sql)

,avg_total
0,5.651942


In [17]:
sql = '''
Select InvoiceId, CustomerId, Total
From Invoice 
where Total > (SELECT Avg(Total) From Invoice)
Order by Total DESC
Limit 4
'''
q(conn, sql)

,InvoiceId,CustomerId,Total
0,404,6,25.86
1,299,26,23.86
2,96,45,21.86
3,194,46,21.86


In [18]:
sql = '''
SELECT c.Country, avg(i.Total)

FROM Customer c
JOIN Invoice i on c.CustomerId = i.CustomerId
Group By c.Country
'''
q(conn, sql)

,Country,avg(i.Total)
0,Argentina,5.374286
1,Australia,5.374286
2,Austria,6.088571
3,Belgium,5.374286
4,Brazil,5.431429
5,Canada,5.427857
6,Chile,6.660000
7,Czech Republic,6.445714
8,Denmark,5.374286
9,Finland,5.945714


In [19]:
sql = '''
SELECT c.Country, i.Total,
    AVG(i.Total) OVER (PARTITION BY c.Country) as country_avg

FROM Customer c
JOIN Invoice i on c.CustomerId = i.CustomerId
'''
q(conn, sql)

,Country,Total,country_avg
0,Argentina,1.98,5.374286
1,Argentina,3.96,5.374286
2,Argentina,5.94,5.374286
3,Argentina,0.99,5.374286
4,Argentina,1.98,5.374286
...,...,...,...
407,United Kingdom,1.98,5.374286
408,United Kingdom,1.98,5.374286
409,United Kingdom,3.96,5.374286
410,United Kingdom,13.86,5.374286


In [20]:
sql = '''
SELECT Total, 
    ROW_NUMBER() OVER (ORDER BY Total DESC) as row_num,
    RANK() OVER (ORDER BY Total DESC) as rnk,
    DENSE_RANK() OVER (ORDER BY Total DESC) as dense
From Invoice
Limit 10
'''
q(conn, sql)

,Total,row_num,rnk,dense
0,25.86,1,1,1
1,23.86,2,2,2
2,21.86,3,3,3
3,21.86,4,3,3
4,18.86,5,5,4
5,18.86,6,5,4
6,17.91,7,7,5
7,16.86,8,8,6
8,16.86,9,8,6
9,15.86,10,10,7


In [21]:
# Customer join Invoice 
# 누적 구매액이 45를 넘는 고객 (이름, 금액), 큰 순 정렬) 
# Customer.FirstName, Customer.LastName, 여태까지 얼마나 총 썼는지
sql = '''
Select c.FirstName, c.LastName, Round(Sum(i.Total), 2) as total_spent
From Customer c
Join Invoice i on c.CustomerId = i.CustomerId
Group By c.CustomerId
Having sum(i.Total) > 45
Order by total_spent DESC
'''
q(conn, sql)

,FirstName,LastName,total_spent
0,Helena,Holý,49.62
1,Richard,Cunningham,47.62
2,Luis,Rojas,46.62
3,Ladislav,Kovács,45.62
4,Hugh,O'Reilly,45.62
